# Pseudotime segments on the UMAP — AbbasTFScreen

Visualize the **50 quantile pseudotime segments** (the groups used to re-call peaks along the
trajectory) on the multiome UMAP stored in `ComboScreen_processed.h5ad`.
Reads only `obsm/X_umap` + `obs/dpt_pseudotime` + `obs/final_label` (no need to load the big X matrix).

In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

H5AD = "/home/eraslab1/Projects/AbbasTFScreen/ComboScreen_processed.h5ad"
# adaptive EVEN-TRAJECTORY binning (same scheme as the peak calling):
# equal-width pseudotime steps, adjacent bins merged forward until >= BIN_FLOOR cells
BIN_WIDTH = 0.02
BIN_FLOOR = 2000

In [ ]:
f = h5py.File(H5AD, "r")
umap = f["obsm"]["X_umap"][:]                       # cells x 2
pt   = f["obs"]["dpt_pseudotime"][:]
d = f["obs"]["final_label"]                          # categorical
cats = [c.decode() if isinstance(c, bytes) else c for c in d["categories"][:]]
state = np.array([cats[c] if c >= 0 else "NA" for c in d["codes"][:]])
f.close()
print("cells:", umap.shape[0], "| pt range:", round(float(pt.min()),3), "-", round(float(pt.max()),3))

In [ ]:
# adaptive even-trajectory segments (match 09_PseudotimePeaks): equal-width
# pseudotime steps merged forward to a cell floor -> ~16 bins tiling the trajectory
maxb = int(np.floor(pt.max() / BIN_WIDTH))
b = np.minimum((pt // BIN_WIDTH).astype(int), maxb)          # equal-width fine bins
cnt = np.bincount(b, minlength=maxb + 1)
newid = np.zeros(maxb + 1, dtype=int); cur = 1; acc = 0
for i in range(maxb + 1):
    newid[i] = cur; acc += cnt[i]
    if acc >= BIN_FLOOR: cur += 1; acc = 0
if acc > 0 and cur > 1: newid[newid == cur] = cur - 1        # fold trailing small bin
seg = newid[b]
N_SEG = int(seg.max())
counts = np.bincount(seg)[1:]
print(f"{N_SEG} even-trajectory segments; cells/segment {counts.min()}-{counts.max()}")

### UMAP colored by continuous pseudotime

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.2))
o = np.argsort(pt)                                   # draw high-pt on top
s = ax.scatter(umap[o,0], umap[o,1], c=pt[o], cmap="viridis", s=1, rasterized=True)
plt.colorbar(s, label="dpt_pseudotime")
ax.set_title("UMAP — pseudotime (NE=0 -> Differentiated=1)")
ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### UMAP with the main trajectory overlaid
DPT is an ordering, not an explicit curve, so the "main trajectory" is drawn as the running-median UMAP position across 80 pseudotime bins, connected from the NE root to the Differentiated tip.

In [ ]:
# --- main trajectory: median UMAP position along pseudotime bins, connected root->tip ---
NB = 80
o  = np.argsort(pt, kind="mergesort")
tb = np.empty(len(pt), int); tb[o] = (np.arange(len(pt)) * NB) // len(pt)
cx = np.array([np.median(umap[tb==b, 0]) for b in range(NB)])
cy = np.array([np.median(umap[tb==b, 1]) for b in range(NB)])

def rollmed(a, w=5):                     # light smoothing, endpoints preserved
    h = w // 2
    return np.array([np.median(a[max(0,i-h):i+h+1]) for i in range(len(a))])
sx, sy = rollmed(cx), rollmed(cy)

fig, ax = plt.subplots(figsize=(6.8, 5.6))
o2 = np.argsort(pt)
im = ax.scatter(umap[o2,0], umap[o2,1], c=pt[o2], cmap="viridis", s=1, alpha=0.45, rasterized=True)
plt.colorbar(im, label="dpt_pseudotime")
ax.plot(sx, sy, color="black", lw=4.5, solid_capstyle="round", zorder=5)   # trajectory
ax.plot(sx, sy, color="white", lw=1.5, solid_capstyle="round", zorder=6)   # inner highlight
ax.scatter(sx[0], sy[0], s=130, color="black", edgecolor="white", lw=1.5, zorder=7)  # root
ax.annotate("", xy=(sx[-1], sy[-1]), xytext=(sx[-5], sy[-5]),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=4), zorder=7)          # direction
ax.text(sx[0]+0.3, sy[0], "root (NE)", fontsize=9, va="center", weight="bold")
ax.set_title("UMAP — main pseudotime trajectory (median path, NE -> Differentiated)")
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
plt.tight_layout(); plt.show()

### UMAP colored by the pseudotime segments
Sequential palette = ordering (essentially the pseudotime bands); the contrasting palette makes the individual segment boundaries visible.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
# (a) ordered (viridis) -> shows the trajectory direction
o = np.argsort(seg)
s0 = axs[0].scatter(umap[o,0], umap[o,1], c=seg[o], cmap="viridis", s=1, rasterized=True)
plt.colorbar(s0, ax=axs[0], label="segment (1=root ... 50)")
axs[0].set_title(f"{N_SEG} pseudotime segments (ordered)")
# (b) contrasting palette -> shows discrete bands
base = plt.cm.tab20(np.linspace(0,1,20))
cmap = ListedColormap(np.vstack([base, base, base])[:N_SEG])
axs[1].scatter(umap[:,0], umap[:,1], c=seg, cmap=cmap, s=1, rasterized=True)
axs[1].set_title(f"{N_SEG} pseudotime segments (bands)")
for a in axs: a.set_xticks([]); a.set_yticks([]); a.set_xlabel("UMAP1"); a.set_ylabel("UMAP2")
plt.tight_layout(); plt.show()

### UMAP colored by cell state (reference)

In [ ]:
order_states = ["Neuroendocrine","Intermediate-3","Intermediate-1","Intermediate-2",
                "Differentiated-2","Differentiated_1"]
order_states = [s for s in order_states if s in set(state)] + [s for s in sorted(set(state)) if s not in order_states]
fig, ax = plt.subplots(figsize=(6.8, 5.2))
cmap = plt.cm.tab10
for i, s in enumerate(order_states):
    m = state == s
    ax.scatter(umap[m,0], umap[m,1], s=1, color=cmap(i % 10), label=f"{s} (n={int(m.sum())})", rasterized=True)
ax.legend(markerscale=6, fontsize=7, loc="best", frameon=False)
ax.set_title("UMAP — cell state (final_label)")
ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("UMAP1"); ax.set_ylabel("UMAP2")
plt.tight_layout(); plt.show()

### Segment summary (every 5th segment)

In [ ]:
print(f"{'seg':>4} {'n':>7} {'pt_min':>9} {'pt_max':>9}   dominant state")
for sgi in list(range(1, N_SEG+1, 5)) + [N_SEG]:
    m = seg == sgi
    vals, cnts = np.unique(state[m], return_counts=True)
    dom = vals[cnts.argmax()]
    print(f"{sgi:>4} {int(m.sum()):>7} {pt[m].min():>9.4f} {pt[m].max():>9.4f}   {dom}")

### Distribution of pseudotime across the 16 segments
Left: pseudotime density (log-y, NE-skewed) with the segment boundaries — closely spaced where cells are dense (NE), wider across the sparse differentiated tail. Right: the pseudotime range each segment covers.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))

# (A) pseudotime density with the segment boundaries
edges = [pt[seg==s].min() for s in range(1, N_SEG+1)] + [pt.max()]
axs[0].hist(pt, bins=200, color="lightgray")
for e in edges:
    axs[0].axvline(e, color="crimson", lw=0.7, alpha=0.8)
axs[0].set_yscale("log")
axs[0].set_xlabel("dpt_pseudotime"); axs[0].set_ylabel("# cells (log)")
axs[0].set_title(f"pseudotime density + {N_SEG} segment boundaries")

# (B) pseudotime distribution per segment
data = [pt[seg==s] for s in range(1, N_SEG+1)]
bp = axs[1].boxplot(data, showfliers=False, widths=0.7, patch_artist=True)
for i, box in enumerate(bp["boxes"]):
    box.set_facecolor(plt.cm.viridis(i/(N_SEG-1))); box.set_alpha(0.85)
axs[1].set_xlabel("segment (1 = NE root ... %d = Differentiated)" % N_SEG)
axs[1].set_ylabel("dpt_pseudotime")
axs[1].set_title("pseudotime distribution per segment")
axs[1].tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

### Notes
- Segments are pseudotime **quantile** bins (equal cells, ~5,600 each), so early bins are narrow in pseudotime (dense NE) and late bins span wide pseudotime (sparse Differentiated) — visible as unevenly-sized bands on the UMAP.
- This UMAP is the full multiome embedding (291,301 cells). The peak-calling used the 280,177 ATAC cells binned identically; the picture is the same up to the ~4% ATAC-only difference.